# 03 — Football Knowledge Layer: Feature Engineering

**CRISP-DM phase:** Data Preparation / early Modelling.

Implements the ontology in `outputs/football_knowledge/football_ontology.md`: raw per-90 metrics -> football concepts -> tactical dimensions, using the hybrid aggregation method decided in `outputs/PHASE0_DELIVERABLE.md` (OD-4) -- PCA checks that hand-grouped concept inputs cohere statistically; explicit, argued weights then combine concepts into dimensions.

**Population baseline (added after a v1 sanity check surfaced a real confound):** the first pass z-scored concept inputs against only the 4-6 clubs in our manager sample, which mechanically rewarded Manchester City/Liverpool players over Sassuolo/Benevento players on raw volume regardless of managerial intent -- exactly the team-strength/team-weakness confound flagged in `manager_philosophy_hypotheses.md` SS4.4. This version instead standardises every concept input against the full big-5-league population (ENG/ITA/FRA, ~30 teams each, 2017/18-2023/24) within the same position group, so a high score means genuinely unusual for the position across the league, not merely above-average within our small manager sample.

This is v1: 9 concepts, 7 of the 13 player-constructible tactical dimensions (TD1, TD3, TD4, TD6, TD7, TD9, TD11). TD5, TD8 (as standalone), TD10 and TD13 need team-level or opponent-relative context not yet built, and are deferred -- noted here rather than silently skipped.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from football_features import (
    attach_possession_adjustment,
    CONCEPTS,
    DIMENSION_WEIGHTS,
    build_concept_scores,
    build_dimension_scores,
    build_manager_profiles,
    build_player_profiles,
    build_population_baseline,
    position_group,
)
from data_sources import DATA_PROCESSED

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

player_match = pd.read_parquet(DATA_PROCESSED / "player_match_manager.parquet")
player_match.shape

(10996, 267)

## 1. Position groups

Per brief SS21-22: never compare a striker's passing volume to a midfielder's on a global z-score. Every concept below is standardised **within position group**, against the full-league population (SS2 below).

In [2]:
player_match = attach_possession_adjustment(player_match, team_col="team", league_code_col="league_code")
player_match["pos_group"] = position_group(player_match)
player_match.pos_group.value_counts()

pos_group
Winger/Second Striker    3308
Full-back/Wing-back      1862
Centre-back              1764
Striker                  1256
Central Midfielder        888
Goalkeeper                780
Defensive Midfielder      660
Attacking Midfielder      478
Name: count, dtype: int64

## 2. League-wide population baseline

Loads the full ENG/ITA/FRA raw match-level files (all ~30 teams per league, not just our 6 in-scope clubs) to compute a per-90, position-group mean/std for every concept-input metric. This takes a minute or two -- it is reading the full-league CSVs, not the manager-attributed subset.

In [3]:
population_baseline = build_population_baseline()
print(population_baseline.shape)
population_baseline.head(10)

(216, 6)


,concept,term,pos_group,mean,std,n
0,aerial_play,t0,Attacking Midfielder,1.174152,4.931995,12411.0
1,aerial_play,t0,Central Midfielder,1.326902,4.141811,27080.0
2,aerial_play,t0,Centre-back,2.422513,3.991909,37925.0
3,aerial_play,t0,Defensive Midfielder,1.458174,3.817145,11918.0
4,aerial_play,t0,Full-back/Wing-back,1.310705,3.400012,35478.0
5,aerial_play,t0,Goalkeeper,0.240725,0.587116,14961.0
6,aerial_play,t0,Striker,2.673352,6.229042,31597.0
7,aerial_play,t0,Winger/Second Striker,1.140522,4.491199,42722.0
8,aerial_play,t1,Attacking Midfielder,34.444326,38.345798,7675.0
9,aerial_play,t1,Central Midfielder,48.977993,39.634906,18603.0


## 3. Concept scores + PCA coherence check (OD-4)

For each football concept, `variance_explained_pc1` is the share of variance in its hand-grouped input metrics explained by their first principal component (computed on our manager sample, since that is the population we are actually profiling -- the *coherence* check and the *standardisation baseline* are two different things and are allowed to use different populations). A high value means the metrics we grouped together under one football label really do move together statistically.

In [4]:
concept_scores, pca_diag = build_concept_scores(player_match, population_baseline=population_baseline)
pca_df = pd.DataFrame(pca_diag).T
pca_df

,n_inputs,variance_explained_pc1
progressive_passing,4.0,0.422
progressive_carrying,3.0,0.612
possession_retention,3.0,0.425
chance_creation,2.0,0.750
box_presence,1.0,NaN
pressing_intensity,5.0,0.405
individual_defending,2.0,0.645
aerial_play,2.0,0.643
crossing,2.0,0.701
goalkeeper_distribution,3.0,0.665


## 4. Tactical dimension scores

Explicit weights from `src/football_features.py::DIMENSION_WEIGHTS`, argued in `outputs/football_knowledge/football_ontology.md`.

In [5]:
dim_scores = build_dimension_scores(concept_scores)
dim_scores.describe().T

,count,mean,std,min,25%,50%,75%,max
TD1_buildup_directness,8901.0,0.257187,0.508694,-2.859538,-0.001312,0.256417,0.515652,6.902470
TD3_progression_passing,8830.0,0.167497,0.564424,-2.499654,-0.141981,0.119166,0.415650,10.781890
TD4_progression_carrying,8605.0,-0.005869,0.653846,-0.734388,-0.349202,-0.104456,0.171906,14.138536
TD6_chance_creation,8863.0,-0.028331,0.607761,-0.901569,-0.353571,-0.179576,0.088616,17.998193
TD7_box_presence,8674.0,0.072808,0.921158,-1.125646,-0.568563,-0.101085,0.367347,14.460066
TD9_pressing_intensity,10996.0,0.112316,0.738040,-0.787538,-0.200025,-0.014626,0.240545,23.563477
TD11_individual_defending,10996.0,-0.142504,0.691309,-1.773890,-0.591320,-0.273703,0.157412,20.265315


## 5. Player profiles (minutes-weighted, per manager-season)

In [6]:
player_profiles = build_player_profiles(player_match, concept_scores, dim_scores)
print(player_profiles.shape)
player_profiles.sort_values("total_minutes", ascending=False).head(10)

(2711, 24)


,Player,Player_Href,manager,team,Season_End_Year,Pos,progressive_passing,progressive_carrying,possession_retention,chance_creation,box_presence,pressing_intensity,individual_defending,aerial_play,crossing,goalkeeper_distribution,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD9_pressing_intensity,TD11_individual_defending,total_minutes
2596,Virgil van Dijk,/en/players/e06683ca/Virgil-van-Dijk,Jürgen Klopp,Liverpool,2020,CB,0.087973,-0.138981,0.679289,-0.091434,-0.027651,-0.042090,-0.545384,0.528175,-0.194417,NaN,0.309716,0.087973,-0.138981,-0.122329,-0.027651,-0.042090,-0.115405,3420.0
648,Ederson,/en/players/3bb7b8b4/Ederson,Pep Guardiola,Manchester City,2019,GK,0.328899,-0.045745,0.270662,0.519658,-0.053905,0.820301,0.218326,-0.074045,-0.015272,0.435284,0.332705,0.328899,-0.045745,0.359179,-0.053905,0.820301,0.130615,3420.0
143,Alisson,/en/players/7a2e46a8/Alisson,Jürgen Klopp,Liverpool,2019,GK,0.099485,0.219161,0.405193,-0.101184,-0.053905,0.301126,0.136101,-0.109936,-0.015272,0.415322,0.254365,0.099485,0.219161,-0.075410,-0.053905,0.301126,0.062290,3420.0
2595,Virgil van Dijk,/en/players/e06683ca/Virgil-van-Dijk,Jürgen Klopp,Liverpool,2019,CB,0.061883,-0.120152,0.620437,-0.125808,0.178483,0.001264,-0.276982,0.411254,-0.205170,NaN,0.271341,0.061883,-0.120152,-0.149616,0.178483,0.001264,-0.070511,3384.0
173,Andrew Robertson,/en/players/2e4f5f03/Andrew-Robertson,Jürgen Klopp,Liverpool,2021,LB,0.083646,-0.109659,0.353259,0.086285,0.122930,-0.007037,-0.500808,-0.092243,0.325100,NaN,0.184751,0.083646,-0.109659,0.157929,0.122930,-0.007037,-0.378239,3383.0
651,Ederson,/en/players/3bb7b8b4/Ederson,Pep Guardiola,Manchester City,2022,GK,0.306733,-0.045745,0.412044,0.005957,-0.053905,0.148828,-0.047345,-0.225014,-0.014601,0.522072,0.381394,0.306733,-0.045745,-0.000210,-0.053905,0.148828,-0.104679,3330.0
160,Andrea Consigli,/en/players/8cabdc1e/Andrea-Consigli,Roberto De Zerbi,Sassuolo,2021,GK,NaN,NaN,NaN,NaN,NaN,0.115790,1.294835,0.034466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.115790,0.098973,3330.0
147,Alisson,/en/players/7a2e46a8/Alisson,Jürgen Klopp,Liverpool,2023,GK,0.220996,-0.045745,0.724329,-0.037695,-0.053905,0.145180,-0.238610,-0.090058,-0.015188,0.571718,0.442140,0.220996,-0.045745,-0.030943,-0.053905,0.145180,-0.195060,3330.0
650,Ederson,/en/players/3bb7b8b4/Ederson,Pep Guardiola,Manchester City,2021,GK,0.234954,-0.045745,0.320427,0.005154,-0.053905,0.207410,0.055444,-0.243018,-0.015272,0.597508,0.333106,0.234954,-0.045745,-0.000974,-0.053905,0.207410,-0.034095,3240.0
146,Alisson,/en/players/7a2e46a8/Alisson,Jürgen Klopp,Liverpool,2022,GK,0.212327,-0.045745,0.723663,-0.016739,-0.053905,0.336724,-0.012544,0.156941,-0.014583,0.548252,0.432913,0.212327,-0.045745,-0.016092,-0.053905,0.336724,0.013486,3240.0


## 6. Manager profiles

The manager profile is the minutes-weighted mean of every player's dimension scores under that manager across the in-scope sample -- i.e. built from measured player behaviour (now standardised against the full league, not just our sample), not a hand-picked label (brief SS10-11). This is the first real, checkable test of the falsifiable hypotheses in `outputs/football_knowledge/manager_philosophy_hypotheses.md` SS4.2.

In [7]:
dim_cols = list(DIMENSION_WEIGHTS.keys())
manager_profiles = build_manager_profiles(player_profiles, dim_cols)
manager_profiles.set_index("manager").round(3)

,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD9_pressing_intensity,TD11_individual_defending
manager,,,,,,,
Jürgen Klopp,0.224,0.164,-0.044,-0.019,0.059,0.108,-0.147
Pep Guardiola,0.331,0.222,0.023,-0.024,0.069,0.123,-0.205
Roberto De Zerbi,0.203,0.130,-0.020,0.012,0.056,-0.006,-0.086


## 7. Quick hypothesis check

H-P1 (strongly expected): Klopp's pressing intensity > Guardiola's and De Zerbi's.
H-P3 (uncertain/contested, genuinely testable): De Zerbi's pressing intensity may be LOW despite his attacking reputation, because his mechanism operates in-possession, not out-of-possession.
H-B1/H-B3 (expected): Guardiola/De Zerbi show higher build-up directness (possession pole) than Klopp.

This is a sanity check on the pipeline, not the final football-validation pass (that is Task #4).

In [8]:
check_cols = ["TD9_pressing_intensity", "TD1_buildup_directness", "TD3_progression_passing", "TD4_progression_carrying", "TD6_chance_creation", "TD7_box_presence", "TD11_individual_defending"]
manager_profiles.set_index("manager")[check_cols].round(3)

,TD9_pressing_intensity,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD11_individual_defending
manager,,,,,,,
Jürgen Klopp,0.108,0.224,0.164,-0.044,-0.019,0.059,-0.147
Pep Guardiola,0.123,0.331,0.222,0.023,-0.024,0.069,-0.205
Roberto De Zerbi,-0.006,0.203,0.130,-0.020,0.012,0.056,-0.086


## 8. Save processed outputs

In [9]:
player_profiles.to_parquet(DATA_PROCESSED / "player_profiles.parquet", index=False)
manager_profiles.to_parquet(DATA_PROCESSED / "manager_profiles.parquet", index=False)
population_baseline.to_parquet(DATA_PROCESSED / "population_baseline.parquet", index=False)
pca_df.to_csv(Path.cwd().parent / "outputs" / "tables" / "concept_pca_diagnostics.csv")
print("saved")

saved
